# SPDR PIPELINE

# EXTRACT

In [46]:


import os
import time
import yfinance as yf
import pandas as pd
import duckdb
import warnings
from tqdm.auto import tqdm
from datetime import date

start_time = time.time()
warnings.filterwarnings('ignore')
today = date.today().strftime('%Y%m%d')

# # # # # # # # # # # # # # # # # # # #

dir_holdings       = 'Holdings'
dir_composites     = 'Composites'
dir_constituents   = 'Constituents'
dir_options_comp   = 'Options/Composites'
dir_options_consti = 'Options/Constituents'
# script_dir         = os.path.dirname(os.path.abspath(__file__))
# dir_holdings       = os.path.join(script_dir, dir_holdings)
# dir_composites     = os.path.join(script_dir, dir_composites)
# dir_constituents   = os.path.join(script_dir, dir_constituents)
# dir_options_comp   = os.path.join(script_dir, dir_options_comp)
# dir_options_consti = os.path.join(script_dir, dir_options_consti)
os.makedirs(dir_holdings,       exist_ok=True)
os.makedirs(dir_composites,     exist_ok=True)
os.makedirs(dir_constituents,   exist_ok=True)
os.makedirs(dir_options_comp,   exist_ok=True)
os.makedirs(dir_options_consti, exist_ok=True)

retries = 3
retry_pause = 5

etfs = [
    # state street etfs
     'XLC'#, 'XLY', 'XLP', 'XLE', 'XLF', 'XLV', 'XLI', 'XLB', 'XLRE', 'XAR',
    # 'KBE', 'XBI', 'KCE', 'XHE', 'XHS', 'XHB', 'KIE', 'XME', 'XES', 'XOP', 
    # 'XPH', 'KRE', 'XRT', 'XSD', 'XSW', 'XTL', 'XTN', 'XLK', 'XLU', 'SPY'
]

# # # # # # # # # # # # # # # # # # # #

def download_holdings(ticker, path):

    for attempt in range(retries):
        try:    
            ssga = f'https://www.ssga.com/us/en/intermediary/library-content/products/fund-data/etfs/us/holdings-daily-us-en-{ticker.lower()}.xlsx'
            duckdb.sql(f"""
                copy (
                    select *
                    from read_xlsx('{ssga}', range = 'A5:H')
                    where Ticker is not null
                ) to '{dir_holdings}/SSGA_{ticker}.parquet'
            """)
            break
        except Exception as e:
            if attempt == 2:
                print(f"     {ticker} failed: {e}")
                break
            time.sleep(retry_pause)

def download_price(ticker, path, interval, period):

    for attempt in range(retries):
        try:
            price = yf.download(ticker, interval=interval, period=period, multi_level_index=False, progress=False).reset_index()
            price.columns = price.columns.str.lower()
            price.insert(0, 'symbol', ticker)
            price.to_parquet(f'{path}/{ticker}_price_{interval}.parquet')
            break
        except Exception as e:
            if attempt == 2:
                print(f"     {ticker} failed: {e}")
                break
            time.sleep(retry_pause)
                
def download_metadata(ticker, path):

    for attempt in range(retries):
        try:
            metadata = pd.DataFrame(pd.Series(yf.Ticker(ticker).info))
            metadata = metadata.T.set_index('symbol').reset_index()
            metadata.to_parquet(f'{path}/{ticker}_meta.parquet')
            break
        except Exception as e:
            if attempt == 2:
                print(f"     {ticker} failed: {e}")
                break
            time.sleep(retry_pause)

def download_financials(ticker, path):

    for attempt in range(retries):
        try:
            yf.Ticker(ticker).ttm_income_stmt.T.reset_index().to_parquet(f'{path}/{ticker}_ttm_income.parquet')
            yf.Ticker(ticker).ttm_cashflow.T.reset_index().to_parquet(f'{path}/{ticker}_ttm_cashflow.parquet')
            yf.Ticker(ticker).quarterly_income_stmt.T.reset_index().to_parquet(f'{path}/{ticker}_qtr_income.parquet')
            yf.Ticker(ticker).quarterly_cashflow.T.reset_index().to_parquet(f'{path}/{ticker}_qtr_cashflow.parquet')
            yf.Ticker(ticker).quarterly_balance_sheet.T.reset_index().to_parquet(f'{path}/{ticker}_qtr_assets.parquet')
            yf.Ticker(ticker).earnings_dates.reset_index().to_parquet(f'{path}/{ticker}_dates.parquet')
            break
        except Exception as e:
            if attempt == 2:
                print(f"     {ticker} failed: {e}")
                break
            time.sleep(retry_pause)
            
def download_options(ticker, path):

    for attempt in range(retries):
        try:
            chain = yf.Ticker(ticker).option_chain()
            chain.calls.to_parquet(f'{path}/{ticker}_calls_{today}.parquet')
            chain.puts.to_parquet(f'{path}/{ticker}_puts_{today}.parquet')
            break
        except Exception as e:
            if attempt == 2:
                print(f"     {ticker} failed: {e}")
                break
            time.sleep(retry_pause)
        
# # # # # # # # # # # # # # # # # # # #

print('SPDR PIPELINE\n')

print('Composites:\n')

#
path = dir_holdings
print('     Downloading holdings')
for ticker in tqdm(etfs):
    download_holdings(ticker, path)

#
path = dir_composites
print('     Downloading price')
for ticker in tqdm(etfs):
    download_price(ticker, path, '1h', '2y')  
    download_price(ticker, path, '1d', '5y')  
    download_price(ticker, path, '1wk', 'max')
print('     Downloading metadata')
for ticker in tqdm(etfs):
    download_metadata(ticker, path)

#
path = dir_options_comp
print('     Downloading option chain')
for ticker in tqdm(etfs):
    download_options(ticker, path)

# # # # # # # # # # # # # # # # # # # #

print('\nConstituents:\n')

#
get_uniques = duckdb.sql(f"""
    with 
    all_holdings as (
        select
            split_part(
                split_part(filename, '_', 2), '.', 1) as etf,
            Ticker as ticker, 
            Name as name, 
            SEDOL as sedol, 
            Weight as weight, 
            "Local Currency" as currency
        from read_parquet('{dir_holdings}/SSGA*', union_by_name=True)
        where
            sedol != '-' and
            weight > 0
    )
    select
        distinct ticker
    from all_holdings
""").fetchdf()
uniques = get_uniques['ticker'].tolist()

#
path = dir_constituents
print('     Downloading price')
for ticker in tqdm(uniques):
    download_price(ticker, path, '1h', '2y')  
    download_price(ticker, path, '1d', '5y')  
    download_price(ticker, path, '1wk', 'max')
print('     Downloading metadata')
for ticker in tqdm(uniques):
    download_metadata(ticker, path)
print('     Downloading financials')
for ticker in tqdm(uniques):
    download_financials(ticker, path)

#
path = dir_options_consti
print('     Downloading option chain')
for ticker in tqdm(uniques):
    download_options(ticker, path)


print(f"\nDone in {time.time()-start_time:.2f}s")
input("\nPress any key to exit.")

SPDR PIPELINE

Composites:


Constituents:



  0%|          | 0/23 [00:00<?, ?it/s]


Done in 3.80s



Press any key to exit. 


''

# TRANSFORM

In [ ]:
# import duckdb

# Composites
compo_price = r'Composites/*price.parquet'
compo_meta  = r'Composites/*meta.parquet'
compo_hold  = r'Composites/*holdings.parquet'
compo_sec   = r'Composites/*sectors.parquet'
# Constituents
consti_price = r'Constituents/*price.parquet'
consti_meta  = r'Constituents/*meta.parquet'

def get_holdings():
    
    query = duckdb.sql(f"""
    select
        etf, symbol, name,
        weight * 100 as weight
    from read_parquet('{compo_hold}', union_by_name=True)
    """).fetchdf().set_index('etf')
    
    return query

def get_weights():
    
    query = duckdb.sql(f"""
    select
        etf,
        sum(weight) * 100 as top10_weight,
        (1 - sum(weight)) * 100 as others
    from read_parquet('{compo_hold}', union_by_name=True)
    group by etf
    """).fetchdf().set_index('etf')
    
    return query

def get_industry_aggs():
    
    query = duckdb.sql(f"""
    select
        sector, 
        industry, 
        count(*) as count, 
        round(try_cast(sum(totalRevenue) as bigint), 0) as revenue, 
        round(try_cast(sum(sharesOutstanding * trailingEps) as bigint), 2) as income,
        round(try_cast(sum(sharesOutstanding * trailingEps) / sum(totalRevenue) * 100 as double), 2) as margin
    from read_parquet('{consti_meta}', union_by_name=True)
    group by sector, industry
    order by margin desc
    """).fetchdf()

    return query

def get_indicators():

    query = duckdb.sql(f"""
    with 
    master_file as (
        with
        previous_day as (
            select symbol, date,
                open, high, low, close,
                lag(open)  over (partition by symbol order by date asc) as pd_open,
                lag(high)  over (partition by symbol order by date asc) as pd_high,
                lag(low)   over (partition by symbol order by date asc) as pd_low,
                lag(close) over (partition by symbol order by date asc) as pd_close,
                volume, cast(volume * close as bigint) as value
            from read_parquet('{consti_price}', union_by_name=True)
        ),
        change as (
            select *,
                ((close / nullif(pd_close, 0)) - 1) * 100 as chg,
                ((open  / nullif(pd_close, 0)) - 1) * 100 as gap,
            from previous_day
        )
        select *,
            avg(chg)        over w AS mean,
            stddev_pop(chg) over w AS stddev
        from change
        window w as (
            partition by symbol order by date
            rows between unbounded preceding and current row
        )
    ),
    indicators as (
        select *,
            (chg - mean) / stddev as z_score
        from master_file
    )
    select *
    from indicators
    order by date desc, value desc
    """).fetchdf().set_index('symbol')
    
    return query

get_holdings().head()

# END

# yfinance resources

In [44]:
# HOLDINGS
    # insider_purchases
    # insider_transactions
    # insider_roster_holders
    # institutional_holders
    # mutualfund_holders

# ANALYSIS
    # revenue_estimate
    # earnings_estimate
    # growth_estimates
    # eps_revisions
    # eps_trend
    # upgrades_downgrades
    # recommendations
    # analyst_price_targets

# STOCKS
    # dividends
    # splits

# SECTORS
    # keys
        # basic-materials
        # communication-services
        # consumer-cyclical
        # consumer-defensive
        # energy
        # financial-services
        # healthcare
        # industrials
        # real-estate
        # technology
        # utilities
    # attributes
        # industries
        # name
        # overview
        # research_reports
        # symbol
        # ticker
        # top_companies
        # top_etfs
        # top_mutual_funds
        # industry-specific
           # sector_name
           # top_growth_companies
           # top_performing_companies

ticker = 'NVDA'
yf.Ticker(ticker).ttm_income_stmt.T.reset_index().to_parquet(f'{ticker}_ttm_income.parquet')
yf.Ticker(ticker).ttm_cashflow.T.reset_index().to_parquet(f'{ticker}_ttm_cashflow.parquet')
yf.Ticker(ticker).quarterly_income_stmt.T.reset_index().to_parquet(f'{ticker}_qtr_income.parquet')
yf.Ticker(ticker).quarterly_cashflow.T.reset_index().to_parquet(f'{ticker}_qtr_cashflow.parquet')
yf.Ticker(ticker).quarterly_balance_sheet.T.reset_index().to_parquet(f'{ticker}_qtr_assets.parquet')
yf.Ticker(ticker).earnings_dates.reset_index().to_parquet(f'{ticker}_dates.parquet')

In [ ]:
# ticker_list = [
#     # state street etfs
#     'XLC', 'XLY', 'XLP', 'XLE', 'XLF', 'XLV', 'XLI', 'XLB', 'XLRE', 'XAR',
#     'KBE', 'XBI', 'KCE', 'XHE', 'XHS', 'XHB', 'KIE', 'XME', 'XES', 'XOP', 
#     'XPH', 'KRE', 'XRT', 'XSD', 'XSW', 'XTL', 'XTN', 'XLK', 'XLU', 'SPY'
# ]
# # ticker_list = ['NVDA', 'TSLA', 'INTC']

# options = pd.DataFrame()
# for ticker in ticker_list:
#     optionchaincalls = yf.Ticker(ticker).option_chain().calls
#     optionchainputs = yf.Ticker(ticker).option_chain().puts
    
#     optionsx = duckdb.sql(f"""
#         with
#         agg as (
#             with 
#             merged as (
#                 select 'calls' as type, * from optionchaincalls
#                 union all
#                 select 'puts' as type, * from optionchainputs
#             ),
#             calls as (
#                 with totals as (
#                     select
#                         type,
#                         impliedVolatility as IV, 
#                         volume, openInterest as OI, 
#                         sum(volume)       over (partition by type) as totalVolume,
#                         sum(openInterest) over (partition by type) as totalInterest
#                     from merged
#                 )
#                 select *,
#                     volume / totalVolume as weightedVol,
#                     OI     / totalInterest as weightedOI 
#                 from totals
#             ),
#             weighted as (
#                 select *,
#                     IV * weightedVol as volWeightIV,
#                     IV * weightedOI as OIWeightIV
#                 from calls
#             )
#             select
#                 type,
#                 sum(volume) as volume,
#                 sum(OI) as OI,
#                 round(sum(volWeightIV), 2) as volWeightIV,
#                 round(sum(OIWeightIV), 2) as OIWeightIV
#             from weighted
#             group by type
#         ),
#         ratio as (
#             select
#                 '{ticker}' as ticker,
#                 round(
#                     max(case when type = 'calls' then volume end) / 
#                     max(case when type = 'puts'  then volume end), 2) as volume,
#                 round(
#                     max(case when type = 'calls' then OI end) / 
#                     max(case when type = 'puts'  then OI end), 2) as OI,
#                 round(
#                     max(case when type = 'calls' then volWeightIV end) / 
#                     max(case when type = 'puts'  then volWeightIV end), 2) as volWeightIV,
#                 round(
#                     max(case when type = 'calls' then OIWeightIV end) / 
#                     max(case when type = 'puts'  then OIWeightIV end), 2) as OIWeightIV
#             from agg
#         )
#         select * from ratio
#     """).fetchdf()

#     options = pd.concat([options, optionsx], ignore_index=True)   

# options